# 01 — Data Cleaning
## Anime Streaming Platform Analytics

**Business context.** We received a raw warehouse export of watch events
(`data/raw/anime_streaming_raw.csv`, ~41.5k rows × 41 columns, one row per
episode watched, 2023-01 → 2026-06). Like every real export, it is messy:
duplicates, missing values, outliers, mixed date formats, numbers stored as
text, and inconsistent category labels. **No metric computed on this file can
be trusted yet.**

**Goal of this notebook**
1. Inspect and profile the raw data.
2. Clean every quality issue — documenting *why* each decision was made.
3. Engineer analysis-ready features (Viewer Segment, Engagement Score, Age Group, …).
4. Export a clean wide table **and** a star schema (`dim_user`, `dim_content`,
   `dim_date`, `fact_watch_events`, `fact_subscriptions`) for SQL and Power BI.
5. Write an audit trail to `docs/cleaning_log.md`.

> Because the dataset is synthetic with a known answer key
> (`data/raw/injection_report.json`), section 4 *proves* the cleaning recovered
> every injected issue — a luxury real projects don't have, and exactly why a
> validation mindset matters.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED = PROJECT_ROOT / "data" / "processed"
DOCS = PROJECT_ROOT / "docs"
EXPORT_DATE = pd.Timestamp("2026-06-30")   # the dataset's "as of" date


def safe_mode(s: pd.Series):
    """Most frequent non-null value; NaN when a group has no values at all."""
    m = s.mode()
    return m.iloc[0] if len(m) else np.nan


df = pd.read_csv(RAW_DIR / "anime_streaming_raw.csv", low_memory=False)
log = {}   # audit trail, written to docs/cleaning_log.md at the end
print(f"Loaded {df.shape[0]:,} rows x {df.shape[1]} columns")

Loaded 41,582 rows x 41 columns


## 1. Inspection

Before touching anything we profile the data. Three questions drive every
cleaning plan: **what type is each column really**, **where are the gaps**,
and **can rows be trusted to be unique**? Cleaning without profiling first is
how analysts silently destroy data.

In [2]:
df.head(3)

,User_ID,Age,Gender,Country,Region,Subscription_Plan,Subscription_Start_Date,Subscription_End_Date,Subscription_Status,Cancellation_Reason,Monthly_Fee,Revenue,Anime_Title,Studio,Genre,Episode_Number,Episode_Length,Watch_Date,Watch_Time_Minutes,Completion_Percentage,Watch_Session,Device,Operating_System,Internet_Type,Language,User_Rating,Like,Share,Download,Watchlist,Search_Source,Recommendation_Clicked,Ad_Shown,Ad_Clicked,Buffering_Time,Session_Duration,Membership_Tenure,Payment_Method,Support_Tickets,Customer_Satisfaction,Referral_Source
0,U00819,27.0,Male,U.S.,North America,Premium,2025-07-27,26-09-2025,Cancelled,Too Expensive,9.99,19.98,Kuroko's Basketball,Production I.G,Sports,73.0,23,2025-09-05 21:13,5.0,27.6,3,Mobile,Android,Fiber,Japanese,3.0,0,0,0,0,Recommendation,1,0,0,4.5,12.0,2,Credit Card,0,7,Advertisement
1,U06794,19.0,Male,Brazil,South America,Basic,2025-08-11,2026-05-12,Cancelled,Not Enough Content,4.99,44.91,Mobile Suit Gundam: The Witch from Mercury,Sunrise,Mecha,19.0,24,2026-01-21 19:10,15.2,63.8,1,Tablet,iPadOS,Fiber,Portuguese,NaN,0,0,0,0,Recommendation,1,0,0,3.5,23.0,9,Credit Card,0,6,Social Media
2,U06201,20.0,Male,USA,North America,Basic,2024-10-22,NaN,Active,NaN,4.99,99.80,My Hero Academia,Bones,Shonen,65.0,24,2025-11-01 06:55,NaN,52.8,2,smart tv,webOS,Satellite,English,8.0,0,1,0,0,Trending,0,0,0,20.4,22.8,20,Credit Card,0,7,Social Media


In [3]:
profile = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "missing_%": (df.isna().mean() * 100).round(2),
    "unique": df.nunique(),
    "example": df.apply(lambda s: s.dropna().iloc[0] if s.notna().any() else None),
})
profile.sort_values("missing", ascending=False)

,dtype,missing,missing_%,unique,example
Subscription_End_Date,str,27866,67.01,1392,26-09-2025
Cancellation_Reason,str,27866,67.01,6,Too Expensive
User_Rating,str,22107,53.16,11,3.0
Payment_Method,str,7790,18.73,5,Credit Card
Age,float64,1256,3.02,84,27.0
Language,str,1255,3.02,8,Japanese
Device,str,1035,2.49,8,Mobile
Genre,str,829,1.99,12,Sports
Country,str,829,1.99,15,U.S.
Operating_System,str,628,1.51,8,Android


In [4]:
print(f"Exact duplicate rows : {df.duplicated().sum():,}")
print(f"Numeric columns      : {df.select_dtypes('number').shape[1]} of {df.shape[1]}")
print("\nSubscription_Plan raw labels:", sorted(df['Subscription_Plan'].astype(str).unique())[:12])
print("Gender raw labels    :", sorted(df['Gender'].dropna().astype(str).unique()))
df[["Age", "Monthly_Fee", "Watch_Time_Minutes", "Buffering_Time"]].describe().round(1)

Exact duplicate rows : 412
Numeric columns      : 18 of 41

Subscription_Plan raw labels: ['BASIC', 'Basic', 'Basic  ', 'FAMILY', 'FREE', 'Family', 'Family  ', 'Free', 'Free  ', 'PREMIUM', 'Premium', 'Premium  ']
Gender raw labels    : ['F', 'Female', 'M', 'Male', 'Other', 'other']


,Age,Watch_Time_Minutes,Buffering_Time
count,40326.0,40958.0,41582.0
mean,29.1,16.8,8.9
std,11.3,108.8,8.3
min,13.0,0.5,-49.0
25%,22.0,11.3,3.3
50%,27.0,14.1,6.1
75%,34.0,17.0,11.8
max,298.0,5885.0,116.0


### What the profile tells us

| Finding | Evidence | Consequence if ignored |
|---|---|---|
| Almost everything loaded as `object` | dtype column above | no math, no time series — `$9.99`, `85.3%`, `12.0` strings poison whole columns |
| ~400 exact duplicate rows | `duplicated()` | every count/sum inflated |
| 8 columns with 1.5–3% missing | missing_% | naive `dropna()` would discard ~15% of rows for no reason |
| Category label chaos | `premium`, `PREMIUM`, `M`, `U.S.` | one plan becomes 4 groups in every `groupby` |
| Impossible values | `Age` max 299, negative buffering, watch time 5,970 min | means/percentiles meaningless |
| Mixed date formats | ISO + `DD/MM/YYYY` + `March 5, 2024` | date parsing silently swaps day/month if guessed |

**Cleaning strategy** (each step below): fix *recoverable* values, null the
unrecoverable, drop rows only when the whole record is untrustworthy — and log
every action.

## 2. Row-level cleaning

### 2.1 Exact duplicates
Duplicate rows double-count revenue, watch time and engagement. These are
full-row copies (same user, same timestamp, same everything), so dropping all
but the first occurrence is safe — no information is lost.

In [5]:
before = len(df)
df = df.drop_duplicates(keep="first").reset_index(drop=True)
log["duplicates_removed"] = before - len(df)
print(f"Removed {log['duplicates_removed']:,} duplicate rows -> {len(df):,} remain")

Removed 412 duplicate rows -> 41,170 remain


### 2.2 Whitespace hygiene
`"  Attack on Titan "` and `"Attack on Titan"` are different strings to a
computer — they split groupbys and break joins to dimension tables. We strip
leading/trailing whitespace from every string cell. (We use `.map` with an
`isinstance` guard because some object columns hold a *mixture* of strings and
numbers — a plain `.str.strip()` would turn the numeric values into NaN.)

In [6]:
str_cols = df.select_dtypes("object").columns
changed = 0
for col in str_cols:
    stripped = df[col].map(lambda v: v.strip() if isinstance(v, str) else v)
    changed += (stripped != df[col]).sum() - (stripped.isna() & df[col].isna()).sum()
    df[col] = stripped
log["whitespace_cells_fixed"] = int(changed)
print(f"Normalised whitespace in {changed:,} cells across {len(str_cols)} text columns")

C:\Users\sound\AppData\Local\Temp\ipykernel_32016\1177932704.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  str_cols = df.select_dtypes("object").columns


Normalised whitespace in 2,624 cells across 23 text columns


### 2.3 Standardising categories
Rule: **one business concept = exactly one label.** We map every variant to a
canonical form and then *assert* the value set — an assert catches variants we
didn't anticipate instead of letting them leak into dashboards.

In [7]:
plan_before = df["Subscription_Plan"].nunique()
df["Subscription_Plan"] = df["Subscription_Plan"].str.title()

df["Gender"] = (df["Gender"].replace({"M": "Male", "F": "Female"}).str.title())

df["Country"] = df["Country"].replace({"United States": "USA", "U.S.": "USA"})

device_map = {"mobile": "Mobile", "smart tv": "Smart TV", "desktop": "Desktop", "tablet": "Tablet"}
df["Device"] = df["Device"].str.lower().map(device_map)

assert set(df["Subscription_Plan"].unique()) == {"Free", "Basic", "Premium", "Family"}
assert set(df["Gender"].dropna().unique()) == {"Male", "Female", "Other"}
assert df["Country"].dropna().nunique() == 13
assert set(df["Device"].dropna().unique()) <= {"Mobile", "Smart TV", "Desktop", "Tablet"}

log["plan_labels_merged"] = f"{plan_before} raw labels -> 4 canonical"
print(f"Subscription_Plan: {plan_before} raw labels -> 4 canonical")
df["Subscription_Plan"].value_counts()

Subscription_Plan: 12 raw labels -> 4 canonical


Subscription_Plan
Premium    16676
Family      9101
Basic       8362
Free        7031
Name: count, dtype: int64

### 2.4 Numbers stored as text
`$9.99`, `85.3%`, `"12.0"` and `"Not Rated"` force whole columns to `object`.
We strip the decoration and coerce with `pd.to_numeric(errors="coerce")` so
anything unconvertible becomes an explicit NaN instead of a crash — then we
*count* the coercions to make sure nothing unexpected was lost.

In [8]:
fixes = {
    "Monthly_Fee": pd.to_numeric(df["Monthly_Fee"].astype(str).str.lstrip("$"), errors="coerce"),
    "Completion_Percentage": pd.to_numeric(
        df["Completion_Percentage"].astype(str).str.rstrip("%"), errors="coerce"),
    "Episode_Number": pd.to_numeric(df["Episode_Number"], errors="coerce").round().astype("Int64"),
    "User_Rating": pd.to_numeric(df["User_Rating"].replace("Not Rated", np.nan),
                                 errors="coerce").round().astype("Int64"),
    "Age": pd.to_numeric(df["Age"], errors="coerce"),
    "Watch_Time_Minutes": pd.to_numeric(df["Watch_Time_Minutes"], errors="coerce"),
    "Buffering_Time": pd.to_numeric(df["Buffering_Time"], errors="coerce"),
    "Revenue": pd.to_numeric(df["Revenue"], errors="coerce"),
    "Membership_Tenure": pd.to_numeric(df["Membership_Tenure"], errors="coerce"),
}
for col, series in fixes.items():
    df[col] = series
log["numeric_columns_recovered"] = len(fixes)
df[list(fixes)].dtypes

Monthly_Fee              float64
Completion_Percentage    float64
Episode_Number             Int64
User_Rating                Int64
Age                      float64
Watch_Time_Minutes       float64
Buffering_Time           float64
Revenue                  float64
Membership_Tenure          int64
dtype: object

### 2.5 Parsing mixed date formats
Three formats coexist in `Watch_Date`: ISO (`2024-03-05 14:30`), day-first
(`05/03/2024 14:30`) and verbose (`March 5, 2024 14:30`). **We never let
pandas guess**: a guessed `05/03/2024` becomes May 3rd instead of March 5th —
a silent, unrecoverable corruption. Instead we detect each format with a regex
and parse each subset with its explicit format string.

In [9]:
def parse_mixed(series: pd.Series, patterns: dict[str, str]) -> pd.Series:
    """Parse a date column where several explicit formats coexist.

    patterns: {regex -> strptime format}. Values matching no pattern -> NaT.
    """
    s = series.astype("string")
    out = pd.Series(pd.NaT, index=series.index, dtype="datetime64[ns]")
    for regex, fmt in patterns.items():
        mask = s.str.match(regex, na=False) & out.isna()
        out[mask] = pd.to_datetime(s[mask], format=fmt)
    return out


df["Watch_Date"] = parse_mixed(df["Watch_Date"], {
    r"\d{4}-\d{2}-\d{2}": "%Y-%m-%d %H:%M",
    r"\d{2}/\d{2}/\d{4}": "%d/%m/%Y %H:%M",
    r"[A-Za-z]+ \d{1,2}, \d{4}": "%B %d, %Y %H:%M",
})
for col in ("Subscription_Start_Date", "Subscription_End_Date"):
    df[col] = parse_mixed(df[col], {
        r"\d{4}-\d{2}-\d{2}": "%Y-%m-%d",
        r"\d{2}-\d{2}-\d{4}": "%d-%m-%Y",
    })

active = df["Subscription_Status"].eq("Active")
print(f"Watch_Date parsed          : {df['Watch_Date'].notna().sum():,} / {len(df):,}")
print(f"Start_Date parsed          : {df['Subscription_Start_Date'].notna().sum():,} / {len(df):,}")
print(f"End_Date parsed (cancelled): {df.loc[~active, 'Subscription_End_Date'].notna().sum():,}"
      f" / {(~active).sum():,}  (Active users legitimately have no end date)")

Watch_Date parsed          : 41,170 / 41,170
Start_Date parsed          : 41,170 / 41,170
End_Date parsed (cancelled): 13,587 / 13,587  (Active users legitimately have no end date)


### 2.6 Impossible values and dates

Decision framework — in order of preference:

| Situation | Action | Rationale |
|---|---|---|
| Value is impossible but *recomputable* from other columns | **fix** | keep the row, keep the signal |
| Impossible and not recomputable | **null it** (impute later) | an honest NaN beats a wrong number |
| The whole record is untrustworthy | **drop the row** | e.g. a watch event dated after the export date — we can't know when (or if) it happened |

Applied here: ages 150–300 → null (a person's age can't be recovered from
other columns); watch time far beyond the episode length → null, then
*recomputed* from `Episode_Length × Completion%`; negative buffering → null;
subscription end **before** start → null the end date (the user-level
consolidation in §3 recovers the true end date from the user's other rows);
future-dated watch events → **dropped**.

In [10]:
# Age: outside plausible human/product range -> null
bad_age = ~df["Age"].between(10, 100) & df["Age"].notna()
df.loc[bad_age, "Age"] = np.nan
log["age_outliers_nulled"] = int(bad_age.sum())

# Watch time: physically capped by episode length (50% tolerance for pauses/rewinds)
bad_wt = df["Watch_Time_Minutes"] > np.maximum(df["Episode_Length"] * 1.5, 60)
df.loc[bad_wt, "Watch_Time_Minutes"] = np.nan
log["watch_time_outliers_nulled"] = int(bad_wt.sum())

# Buffering cannot be negative
bad_buf = df["Buffering_Time"] < 0
df.loc[bad_buf, "Buffering_Time"] = np.nan
log["negative_buffering_nulled"] = int(bad_buf.sum())

# Subscription end before start -> impossible; null and recover in section 3
bad_end = df["Subscription_End_Date"] < df["Subscription_Start_Date"]
df.loc[bad_end, "Subscription_End_Date"] = pd.NaT
log["end_before_start_nulled"] = int(bad_end.sum())

# Watch events after the export date -> the record itself is unreliable
future = df["Watch_Date"] > EXPORT_DATE
df = df[~future].reset_index(drop=True)
log["future_dated_rows_dropped"] = int(future.sum())

pd.Series(log).to_frame("count")

,count
duplicates_removed,412
whitespace_cells_fixed,2624
plan_labels_merged,12 raw labels -> 4 canonical
numeric_columns_recovered,9
age_outliers_nulled,30
watch_time_outliers_nulled,25
negative_buffering_nulled,20
end_before_start_nulled,206
future_dated_rows_dropped,124


### 2.7 Missing values — a strategy per column, never a blanket rule

`dropna()` would throw away thousands of good rows; `fillna(0)` would invent
data. Each column gets the *least destructive* strategy its meaning allows:

| Column | Strategy | Why |
|---|---|---|
| `Genre`, `Studio` | look up from `Anime_Title` | the title fully determines them — **exact recovery** |
| `Device` | infer from `Operating_System`, else user's usual device | `Tizen` can only be a Smart TV |
| `Operating_System` | most common OS for that device | best available estimate, clearly bounded |
| `Watch_Time_Minutes` | recompute `Episode_Length × Completion% / 100` | arithmetic identity, not a guess |
| `Completion_Percentage` | inverse: `Watch_Time / Episode_Length × 100` | same identity ("incomplete sessions") |
| `Buffering_Time` | median *within Internet_Type* | buffering is driven by connection type |
| `User_Rating` | **leave as NaN** | most viewers never rate — imputing would fabricate opinions |
| `Age`, `Country`, `Language`, `Payment_Method` | resolved at *user grain* in §3 | user-level facts should be fixed once per user, not per row |

In [11]:
n0 = df.isna().sum()

# Genre & Studio: exact recovery via title lookup
for col in ("Genre", "Studio"):
    lookup = df.groupby("Anime_Title")[col].agg(safe_mode)
    df[col] = df[col].fillna(df["Anime_Title"].map(lookup))

# Device from OS (unambiguous OSes), then the user's most frequent device
os_device = {"Windows": "Desktop", "macOS": "Desktop", "iPadOS": "Tablet",
             "Tizen": "Smart TV", "webOS": "Smart TV", "Android TV": "Smart TV", "iOS": "Mobile"}
df["Device"] = df["Device"].fillna(df["Operating_System"].map(os_device))
df["Device"] = df["Device"].fillna(df.groupby("User_ID")["Device"].transform(safe_mode))
df["Device"] = df["Device"].fillna("Unknown")

# OS: most frequent OS on that device
df["Operating_System"] = df["Operating_System"].fillna(
    df.groupby("Device")["Operating_System"].transform(safe_mode))

# Watch time <-> completion: mutual arithmetic recovery
wt_gap = df["Watch_Time_Minutes"].isna() & df["Completion_Percentage"].notna()
df.loc[wt_gap, "Watch_Time_Minutes"] = (
    df.loc[wt_gap, "Episode_Length"] * df.loc[wt_gap, "Completion_Percentage"] / 100).round(1)
cp_gap = df["Completion_Percentage"].isna() & df["Watch_Time_Minutes"].notna()
df.loc[cp_gap, "Completion_Percentage"] = (
    df.loc[cp_gap, "Watch_Time_Minutes"] / df.loc[cp_gap, "Episode_Length"] * 100).clip(0, 100).round(1)
# both missing (rare overlap of the two injections): median completion, then derive watch time
both = df["Watch_Time_Minutes"].isna() & df["Completion_Percentage"].isna()
df.loc[both, "Completion_Percentage"] = df["Completion_Percentage"].median()
df.loc[both, "Watch_Time_Minutes"] = (
    df.loc[both, "Episode_Length"] * df.loc[both, "Completion_Percentage"] / 100).round(1)

# Buffering: median within connection type
df["Buffering_Time"] = df["Buffering_Time"].fillna(
    df.groupby("Internet_Type")["Buffering_Time"].transform("median"))

recovered = (n0 - df.isna().sum()).loc[lambda s: s > 0]
log["values_recovered_row_level"] = int(recovered.sum())
print("Values recovered per column:")
recovered.to_frame("recovered")

Values recovered per column:


,recovered
Genre,821
Watch_Time_Minutes,639
Completion_Percentage,614
Device,1027
Operating_System,617
Buffering_Time,20


## 3. User-level consolidation — escaping the grain trap

This table's grain is *watch events*, but 16 columns (age, plan, revenue,
tenure, satisfaction, …) describe the *user*. Denormalised exports corrupt
these independently on each row — one row says a user is 24, another says 250.

The fix: rebuild one authoritative record per user by taking the
**most frequent value across that user's rows** (`mode`). A corrupted value on
1 of a user's 5 rows is outvoted by the 4 clean ones. For the end date we take
the **latest end date that is ≥ the start date** — which also recovers the
end dates we nulled in §2.6. Remaining gaps get user-level fallbacks
(median age by country, dominant language by country, `None` payment for Free
users).

In [12]:
df["_Valid_End"] = df["Subscription_End_Date"].where(
    df["Subscription_End_Date"] >= df["Subscription_Start_Date"])

USER_COLS = ["Age", "Gender", "Country", "Region", "Language", "Subscription_Plan",
             "Subscription_Start_Date", "Subscription_Status", "Cancellation_Reason",
             "Monthly_Fee", "Revenue", "Membership_Tenure", "Payment_Method",
             "Support_Tickets", "Customer_Satisfaction", "Referral_Source"]

users = (df.groupby("User_ID")
           .agg({**{c: safe_mode for c in USER_COLS}, "_Valid_End": "max"})
           .rename(columns={"_Valid_End": "Subscription_End_Date"})
           .reset_index())

# user-level fallbacks for anything still missing
users["Age"] = users["Age"].fillna(users.groupby("Country")["Age"].transform("median"))
users["Age"] = users["Age"].fillna(users["Age"].median()).round().astype(int)
users["Country"] = users["Country"].fillna("Unknown")
country_lang = users.groupby("Country")["Language"].agg(safe_mode)
users["Language"] = users["Language"].fillna(users["Country"].map(country_lang)).fillna("Japanese")
users["Payment_Method"] = users["Payment_Method"].fillna(
    users["Subscription_Plan"].map(lambda p: "None" if p == "Free" else "Unknown"))
for col in ("Membership_Tenure", "Support_Tickets", "Customer_Satisfaction"):
    users[col] = users[col].astype(int)

# end dates recovered by consolidation
still_missing_end = (users["Subscription_Status"].eq("Cancelled")
                     & users["Subscription_End_Date"].isna()).sum()
log["users_consolidated"] = len(users)
log["cancelled_users_missing_end_date"] = int(still_missing_end)

assert users["User_ID"].is_unique
print(f"{len(users):,} users consolidated; "
      f"{still_missing_end} cancelled users still lack an end date")
users.head(3)

7,993 users consolidated; 36 cancelled users still lack an end date


,User_ID,Age,Gender,Country,Region,Language,Subscription_Plan,Subscription_Start_Date,Subscription_Status,Cancellation_Reason,Monthly_Fee,Revenue,Membership_Tenure,Payment_Method,Support_Tickets,Customer_Satisfaction,Referral_Source,Subscription_End_Date
0,U00001,33,Male,Mexico,North America,Spanish,Premium,2023-03-16,Cancelled,Not Enough Content,9.99,109.89,11,Debit Card,3,6,Social Media,2024-02-14
1,U00002,26,Male,South Korea,Asia,Korean,Basic,2026-02-24,Active,NaN,4.99,19.96,4,Debit Card,0,8,YouTube,NaT
2,U00003,26,Male,USA,North America,English,Basic,2025-10-12,Active,NaN,4.99,39.92,8,Credit Card,1,5,Friend Referral,NaT


In [13]:
# Replace per-row user attributes with the canonical user record
events = (df.drop(columns=USER_COLS + ["Subscription_End_Date", "_Valid_End"])
            .merge(users, on="User_ID", how="left"))
events = events.sort_values("Watch_Date").reset_index(drop=True)
print(f"Events table rebuilt: {len(events):,} rows, all user attributes canonical")

Events table rebuilt: 41,046 rows, all user attributes canonical


## 4. Verification against the answer key

The generator logged exactly what it corrupted
(`data/raw/injection_report.json`). We now check that cleaning recovered it.
Counts can differ slightly from the key for two legitimate reasons: ~1% of
rows were *duplicated after* corruption (one injection, two dirty rows), and
injections can *overlap* (a missing-value pass can blank a cell that was
already an outlier, leaving one fewer outlier to find). The zero-tolerance
checks are the ones that matter: no duplicates, no variant labels, no
text-numbers, no impossible dates left.

In [14]:
key = json.loads((RAW_DIR / "injection_report.json").read_text())

recovered_vs_injected = pd.DataFrame([
    ("Duplicate rows removed", log["duplicates_removed"], key["duplicate_rows"]),
    ("Age outliers nulled", log["age_outliers_nulled"], key["age_outliers"]),
    ("Watch-time outliers nulled", log["watch_time_outliers_nulled"], key["watch_time_outliers"]),
    ("Negative buffering nulled", log["negative_buffering_nulled"], key["negative_buffering"]),
    ("End-before-start fixed", log["end_before_start_nulled"], key["end_before_start"]),
    ("Future-dated rows dropped", log["future_dated_rows_dropped"], key["future_watch_dates"]),
], columns=["check", "recovered", "injected"])
recovered_vs_injected["delta"] = recovered_vs_injected["recovered"] - recovered_vs_injected["injected"]

# Zero-tolerance: nothing dirty may remain
assert events.duplicated().sum() == 0
assert set(events["Subscription_Plan"].unique()) == {"Free", "Basic", "Premium", "Family"}
assert events["Monthly_Fee"].isin([0.0, 4.99, 9.99, 14.99]).all()
assert events["Age"].between(10, 100).all()
assert (events["Buffering_Time"] >= 0).all()
assert events["Watch_Date"].notna().all() and (events["Watch_Date"] <= EXPORT_DATE).all()
ended = events["Subscription_End_Date"].notna()
assert (events.loc[ended, "Subscription_End_Date"]
        >= events.loc[ended, "Subscription_Start_Date"]).all()
print("All zero-tolerance assertions passed.\n")
recovered_vs_injected

All zero-tolerance assertions passed.



,check,recovered,injected,delta
0,Duplicate rows removed,412,412,0
1,Age outliers nulled,30,30,0
2,Watch-time outliers nulled,25,25,0
3,Negative buffering nulled,20,20,0
4,End-before-start fixed,206,206,0
5,Future-dated rows dropped,124,124,0


## 5. Feature engineering

Raw columns describe *what happened*; features describe *what it means* for
the business. Each feature answers a recurring stakeholder question:

| Feature | Grain | Business question it answers |
|---|---|---|
| `Age_Group` | user | which demographics do we serve? |
| `Weekend_Viewing`, `Watch_Hour`, `Peak_Hour` | event | when should we schedule releases/ads? |
| `Completion_Bucket` | event | is content being finished or abandoned? |
| `Subscription_Length_Days` | user | how long do we keep customers? |
| `Retention_Status` | user | loyal / retained / early-churn / churned mix |
| `Revenue_Per_Month` (ARPU) | user | what is a customer worth per month? |
| `Engagement_Score` (0–100) | user | single health metric across behaviours |
| `Viewer_Segment` | user | who are our power users vs. light users? |

`Engagement_Score` combines watch frequency, completion, interactions and
rating propensity using **percentile ranks** (robust to outliers and unit-free)
with weights favouring actual watching over clicks: 35% frequency, 30%
completion, 20% interactions, 15% ratings. Segments are score quartiles —
simple, explainable, and stable as the user base grows.

In [15]:
# --- event-level features ---
events["Watch_Hour"] = events["Watch_Date"].dt.hour
events["Peak_Hour"] = events["Watch_Hour"].between(19, 23)
events["Weekend_Viewing"] = events["Watch_Date"].dt.weekday >= 5
events["Completion_Bucket"] = pd.cut(
    events["Completion_Percentage"], bins=[0, 25, 50, 75, 90, 100],
    labels=["Abandoned (<25%)", "Sampled (25-50%)", "Partial (50-75%)",
            "Mostly Watched (75-90%)", "Completed (90-100%)"], include_lowest=True)

# --- user-level features ---
users["Age_Group"] = pd.cut(users["Age"], bins=[0, 17, 24, 34, 44, 120],
                            labels=["13-17", "18-24", "25-34", "35-44", "45+"])
sub_end = users["Subscription_End_Date"].fillna(EXPORT_DATE)
users["Subscription_Length_Days"] = (sub_end - users["Subscription_Start_Date"]).dt.days
users["Retention_Status"] = np.select(
    [users["Subscription_Status"].eq("Active") & (users["Membership_Tenure"] >= 12),
     users["Subscription_Status"].eq("Active"),
     users["Membership_Tenure"] <= 3],
    ["Loyal (12m+ active)", "Active", "Early Churn (<=3m)"], default="Churned")
users["Revenue_Per_Month"] = (users["Revenue"]
                              / users["Membership_Tenure"].clip(lower=1)).round(2)

# --- engagement score: percentile-rank blend of four behaviours ---
events["_any_action"] = events[["Like", "Share", "Download", "Watchlist"]].max(axis=1)
behaviour = events.groupby("User_ID").agg(
    events_total=("User_ID", "size"),
    avg_completion=("Completion_Percentage", "mean"),
    interaction_rate=("_any_action", "mean"),
    rating_rate=("User_Rating", lambda s: s.notna().mean()),
)
behaviour["events_per_month"] = (
    behaviour["events_total"] / users.set_index("User_ID")["Membership_Tenure"].clip(lower=1))
pr = lambda s: s.rank(pct=True)
score = (0.35 * pr(behaviour["events_per_month"]) + 0.30 * pr(behaviour["avg_completion"])
         + 0.20 * pr(behaviour["interaction_rate"]) + 0.15 * pr(behaviour["rating_rate"]))
behaviour["Engagement_Score"] = (score * 100).round(1)
behaviour["Viewer_Segment"] = pd.qcut(
    behaviour["Engagement_Score"], 4,
    labels=["Light Viewer", "Casual Viewer", "Regular Viewer", "Power Viewer"])

users = users.merge(behaviour[["Engagement_Score", "Viewer_Segment"]],
                    left_on="User_ID", right_index=True, how="left")
log["features_engineered"] = 10
users[["Retention_Status", "Viewer_Segment"]].apply(lambda s: s.value_counts())

,Retention_Status,Viewer_Segment
Active,2204.0,NaN
Casual Viewer,NaN,2003.0
Churned,1850.0,NaN
Early Churn (<=3m),2619.0,NaN
Light Viewer,NaN,2001.0
Loyal (12m+ active),1320.0,NaN
Power Viewer,NaN,1994.0
Regular Viewer,NaN,1995.0


In [16]:
# Sanity: do the engineered features behave sensibly?
check = users.groupby("Viewer_Segment", observed=True).agg(
    n=("User_ID", "size"),
    churn_rate=("Subscription_Status", lambda s: s.eq("Cancelled").mean()),
    avg_tenure_months=("Membership_Tenure", "mean"),
    avg_csat=("Customer_Satisfaction", "mean"),
).round(2)
check

,n,churn_rate,avg_tenure_months,avg_csat
Viewer_Segment,,,,
Light Viewer,2001,0.61,7.94,6.66
Casual Viewer,2003,0.59,7.35,6.88
Regular Viewer,1995,0.58,6.65,7.15
Power Viewer,1994,0.45,8.03,7.58


Power Viewers churn far less and are more satisfied than Light Viewers — the
score orders users the way the business would expect, which is exactly what a
composite metric must do before anyone puts it on a dashboard.

## 6. Star schema

One wide table forces every query to fight the grain trap (user attributes
repeated per event). We split it the way a BI team would:

```
                   dim_date                dim_user
                      \                      /
   fact_watch_events ──┼── Content_ID ── dim_content
        (event grain)  │
                       └── User_ID ─── fact_subscriptions (user grain:
                                        plan, dates, revenue, churn)
```

- **Facts** hold measures at a single, explicit grain.
- **Dimensions** hold descriptive attributes once.
- Revenue lives *only* in `fact_subscriptions` — it can never be
  double-counted by an event-level SUM again.
- This model loads straight into SQLite (module 4) and Power BI (module 7).

In [17]:
# dim_content: one row per title
dim_content = (events.groupby("Anime_Title", as_index=False)
    .agg(Studio=("Studio", safe_mode), Genre=("Genre", safe_mode),
         Episodes_Available=("Episode_Number", "max"),
         Avg_Episode_Length=("Episode_Length", "mean")))
dim_content["Avg_Episode_Length"] = dim_content["Avg_Episode_Length"].round(1)
dim_content.insert(0, "Content_ID", range(1, len(dim_content) + 1))

# dim_date: calendar covering the watch window
cal = pd.date_range(events["Watch_Date"].min().normalize(),
                    events["Watch_Date"].max().normalize())
dim_date = pd.DataFrame({"Date": cal})
dim_date["Date_ID"] = dim_date["Date"].dt.strftime("%Y%m%d").astype(int)
dim_date["Year"] = dim_date["Date"].dt.year
dim_date["Quarter"] = dim_date["Date"].dt.quarter
dim_date["Month"] = dim_date["Date"].dt.month
dim_date["Month_Name"] = dim_date["Date"].dt.month_name()
dim_date["Day_Name"] = dim_date["Date"].dt.day_name()
dim_date["Is_Weekend"] = dim_date["Date"].dt.weekday >= 5
dim_date["Is_Season_Launch"] = dim_date["Month"].isin([1, 4, 7, 10])
dim_date = dim_date[["Date_ID", "Date", "Year", "Quarter", "Month",
                     "Month_Name", "Day_Name", "Is_Weekend", "Is_Season_Launch"]]

# dim_user: descriptive user attributes (no revenue measures)
dim_user = users[["User_ID", "Age", "Age_Group", "Gender", "Country", "Region",
                  "Language", "Subscription_Plan", "Subscription_Status",
                  "Viewer_Segment", "Engagement_Score", "Retention_Status",
                  "Referral_Source", "Payment_Method", "Customer_Satisfaction",
                  "Support_Tickets"]].copy()

# fact_subscriptions: user-grain money & lifecycle
fact_subscriptions = users[["User_ID", "Subscription_Plan", "Subscription_Start_Date",
                            "Subscription_End_Date", "Subscription_Status",
                            "Cancellation_Reason", "Monthly_Fee", "Membership_Tenure",
                            "Subscription_Length_Days", "Revenue", "Revenue_Per_Month"]].copy()

# fact_watch_events: event-grain behaviour, keyed to all dimensions
fact_watch_events = events.merge(
    dim_content[["Content_ID", "Anime_Title"]], on="Anime_Title", how="left")
fact_watch_events["Date_ID"] = fact_watch_events["Watch_Date"].dt.strftime("%Y%m%d").astype(int)
fact_watch_events.insert(0, "Event_ID", range(1, len(fact_watch_events) + 1))
fact_watch_events = fact_watch_events[[
    "Event_ID", "User_ID", "Content_ID", "Date_ID", "Watch_Date", "Watch_Hour",
    "Peak_Hour", "Weekend_Viewing", "Episode_Number", "Episode_Length",
    "Watch_Time_Minutes", "Completion_Percentage", "Completion_Bucket",
    "Watch_Session", "Session_Duration", "Device", "Operating_System",
    "Internet_Type", "Buffering_Time", "User_Rating", "Like", "Share",
    "Download", "Watchlist", "Search_Source", "Recommendation_Clicked",
    "Ad_Shown", "Ad_Clicked"]]

for name, frame in {"dim_user": dim_user, "dim_content": dim_content, "dim_date": dim_date,
                    "fact_watch_events": fact_watch_events,
                    "fact_subscriptions": fact_subscriptions}.items():
    print(f"{name:<20} {frame.shape[0]:>7,} rows x {frame.shape[1]:>2} cols")

dim_user               7,993 rows x 16 cols
dim_content               64 rows x  6 cols
dim_date               1,250 rows x  9 cols
fact_watch_events     41,046 rows x 28 cols
fact_subscriptions     7,993 rows x 11 cols


In [18]:
# Referential integrity: every fact key must resolve to a dimension row
assert fact_watch_events["User_ID"].isin(dim_user["User_ID"]).all()
assert fact_watch_events["Content_ID"].isin(dim_content["Content_ID"]).all()
assert fact_watch_events["Date_ID"].isin(dim_date["Date_ID"]).all()
assert fact_subscriptions["User_ID"].isin(dim_user["User_ID"]).all()
assert len(fact_subscriptions) == len(dim_user) == fact_watch_events["User_ID"].nunique()
print("Referential integrity verified: every fact row joins to every dimension.")

Referential integrity verified: every fact row joins to every dimension.


## 7. Export & audit trail

We ship both shapes: the **star schema** (for SQL & Power BI) and one
**clean wide table** (convenient for EDA in module 3). The audit trail goes to
`docs/cleaning_log.md` so anyone can see what was changed without re-running
this notebook.

In [19]:
PROCESSED.mkdir(parents=True, exist_ok=True)
clean_wide = (fact_watch_events
              .merge(dim_user, on="User_ID")
              .merge(dim_content, on="Content_ID")
              .merge(fact_subscriptions.drop(columns=["Subscription_Plan",
                                                      "Subscription_Status"]), on="User_ID"))

exports = {"dim_user": dim_user, "dim_content": dim_content, "dim_date": dim_date,
           "fact_watch_events": fact_watch_events, "fact_subscriptions": fact_subscriptions,
           "anime_streaming_clean": clean_wide}
for name, frame in exports.items():
    path = PROCESSED / f"{name}.csv"
    frame.to_csv(path, index=False)
    print(f"{path.name:<28} {frame.shape[0]:>7,} rows   {path.stat().st_size / 1e6:>5.1f} MB")
log["tables_exported"] = len(exports)

dim_user.csv                   7,993 rows     1.0 MB
dim_content.csv                   64 rows     0.0 MB
dim_date.csv                   1,250 rows     0.1 MB


fact_watch_events.csv         41,046 rows     6.4 MB
fact_subscriptions.csv         7,993 rows     0.6 MB


anime_streaming_clean.csv     41,046 rows    14.6 MB


In [20]:
lines = ["# Cleaning Log", "",
         "*Auto-generated by `notebooks/01_data_cleaning.ipynb`.*", "",
         "| Step | Result |", "|---|---|"]
lines += [f"| {k.replace('_', ' ').capitalize()} | {v} |" for k, v in log.items()]
(DOCS / "cleaning_log.md").write_text("\n".join(lines), encoding="utf-8")
print(f"Wrote {DOCS / 'cleaning_log.md'}")
pd.Series(log).to_frame("result")

Wrote F:\Projects\Vibe coding\AI News summarizer\docs\cleaning_log.md


,result
duplicates_removed,412
whitespace_cells_fixed,2624
plan_labels_merged,12 raw labels -> 4 canonical
numeric_columns_recovered,9
age_outliers_nulled,30
watch_time_outliers_nulled,25
negative_buffering_nulled,20
end_before_start_nulled,206
future_dated_rows_dropped,124
values_recovered_row_level,3738


## 8. Summary

**Before:** 41,582 rows, 41 columns, almost all `object` dtype, ~400
duplicates, eight columns 1.5–3% missing, impossible ages/dates/durations,
four spellings of "Premium".

**After:** a verified star schema — 8,000 users, ~41k trustworthy watch
events, engineered business features, referential integrity guaranteed, and
every cleaning decision logged and cross-checked against the injection answer
key.

**Next:** `02_eda.ipynb` — exploratory analysis of viewer behaviour, revenue,
content and churn on the cleaned data.